# WC_BADGE_DETAILS_D ETL Process
### Dimension Table - SCD Type 1 (Incremental Update with NOT EXISTS Detection)
**ODI Package:** WC_BADGE_DETAILS_D Load

**Source Schema:** `workspace.prxbi_ts_sep`

**Target Schema:** `workspace.prxbi_dw_sep`

**Source Table:** `WC_MERCURY_BADGE_TS`

**Target Table:** `WC_BADGE_DETAILS_D`

**Detection Strategy:** NOT EXISTS (compare all columns to detect changes)

---

In [ ]:
%sql
-- Step 1a: Variable V_ETL_LAST_EXTRACT_TIME
CREATE OR REPLACE TEMP VIEW V_ETL_LAST_EXTRACT_TIME AS
SELECT etl_last_extract_time
FROM workspace.prxbi_dw_sep.wc_etl_parameters
WHERE ETL_JOB_TYPE = 'EOD'

In [ ]:
%sql
-- Step 1b: Variable V_ETL_CURRENT_EXTRACT_TIME
CREATE OR REPLACE TEMP VIEW V_ETL_CURRENT_EXTRACT_TIME AS
SELECT etl_current_extract_time
FROM workspace.prxbi_dw_sep.wc_etl_parameters
WHERE ETL_JOB_TYPE = 'EOD'

In [ ]:
%sql
-- Step 1c: Variable V_ETL_PROC_WID
CREATE OR REPLACE TEMP VIEW V_ETL_PROC_WID AS
SELECT ROW_WID
FROM workspace.prxbi_dw_sep.wc_etl_parameters
WHERE ETL_JOB_TYPE = 'EOD'

---
## Step 2: C$ Work Table - Extract from Source with Deduplication
Extract from `WC_MERCURY_BADGE_TS` with dedup via INNER JOIN on (ID, max INT_INSERT_DATE, max VERSIONNUMBER) within the ETL time window.

In [ ]:
%sql
-- Step 2: C$ Work Table - Source extract with deduplication
CREATE OR REPLACE TEMP VIEW C_WC_BADGE_DETAILS AS
SELECT
    AMERCURY_BADGE_TS.ID AS ID,
    AMERCURY_BADGE_TS.BADGELOCATION AS BADGELOCATION,
    AMERCURY_BADGE_TS.BADGETOKEN AS BADGETOKEN,
    AMERCURY_BADGE_TS.BADGEVERSION AS BADGEVERSION,
    AMERCURY_BADGE_TS.CONTACTEMAIL AS CONTACTEMAIL,
    AMERCURY_BADGE_TS.CONTACTFIRSTNAME AS CONTACTFIRSTNAME,
    AMERCURY_BADGE_TS.CONTACTJOBTITLE AS CONTACTJOBTITLE,
    AMERCURY_BADGE_TS.CONTACTLASTNAME AS CONTACTLASTNAME,
    AMERCURY_BADGE_TS.CONTACTPERSONRXMASTERID AS CONTACTPERSONRXMASTERID,
    AMERCURY_BADGE_TS.CREATEDBYREGISTRATIONTYPE AS CREATEDBYREGISTRATIONTYPE,
    AMERCURY_BADGE_TS.CREATEDBYTYPE AS CREATEDBYTYPE,
    AMERCURY_BADGE_TS.CULTURE AS CULTURE,
    AMERCURY_BADGE_TS.CUSTOMERTYPE AS CUSTOMERTYPE,
    AMERCURY_BADGE_TS.EVENTEDITIONGBSCODE AS EVENTEDITIONGBSCODE,
    AMERCURY_BADGE_TS.ISBADGEUPDATE AS ISBADGEUPDATE,
    AMERCURY_BADGE_TS.MARKETINGPREFERENCESPROMPTREQUIRED AS MARKETINGPREFERENCESPROMPTREQU,
    AMERCURY_BADGE_TS.ORGANISATIONCITY AS ORGANISATIONCITY,
    AMERCURY_BADGE_TS.ORGANISATIONCOUNTRYCODE AS ORGANISATIONCOUNTRYCODE,
    AMERCURY_BADGE_TS.ORGANISATIONDISPLAYNAME AS ORGANISATIONDISPLAYNAME,
    AMERCURY_BADGE_TS.ORGANISATIONRXMASTERID AS ORGANISATIONRXMASTERID,
    AMERCURY_BADGE_TS.ORGANISATIONSTATE AS ORGANISATIONSTATE,
    AMERCURY_BADGE_TS.PARTICIPATINGORGANISATIONID AS PARTICIPATINGORGANISATIONID,
    AMERCURY_BADGE_TS.PRODUCTCODE AS PRODUCTCODE,
    AMERCURY_BADGE_TS.QRCODECONTENT AS QRCODECONTENT,
    AMERCURY_BADGE_TS.REGISTRATIONID AS REGISTRATIONID,
    AMERCURY_BADGE_TS.STATUS AS STATUS,
    AMERCURY_BADGE_TS.SUPPORTSTAFFCOMPANYADDRESS AS SUPPORTSTAFFCOMPANYADDRESS,
    AMERCURY_BADGE_TS.SUPPORTSTAFFCOMPANYNAME AS SUPPORTSTAFFCOMPANYNAME,
    AMERCURY_BADGE_TS.SUPPORTSTAFFMOBILEPHONE AS SUPPORTSTAFFMOBILEPHONE,
    AMERCURY_BADGE_TS.SUPPORTSTAFFREPORTSTO AS SUPPORTSTAFFREPORTSTO,
    AMERCURY_BADGE_TS.SUPPORTSTAFFSTANDS AS SUPPORTSTAFFSTANDS,
    AMERCURY_BADGE_TS.SUPPORTSTAFFUSERACCESS AS SUPPORTSTAFFUSERACCESS,
    AMERCURY_BADGE_TS.VERSIONNUMBER AS VERSIONNUMBER,
    AMERCURY_BADGE_TS.MOBILEPHONE AS MOBILEPHONE,
    AMERCURY_BADGE_TS.FIRSTSCANNEDDATE AS FIRSTSCANNEDDATE,
    AMERCURY_BADGE_TS.LASTPRINTEDDATE AS LASTPRINTEDDATE,
    AMERCURY_BADGE_TS.ACCESSVALIDITYMODIFIEDDATE AS ACCESSVALIDITYMODIFIEDDATE,
    AMERCURY_BADGE_TS.CREATEDDATE AS CREATEDDATE,
    AMERCURY_BADGE_TS.COMPANYPRODUCTCODE AS COMPANYPRODUCTCODE,
    AMERCURY_BADGE_TS.PAYMENTSTATUS AS PAYMENTSTATUS,
    AMERCURY_BADGE_TS.PHOTOKEY AS PHOTOKEY,
    AMERCURY_BADGE_TS.PHOTOSOURCE AS PHOTOSOURCE,
    AMERCURY_BADGE_TS.PHOTOSOURCETYPE AS PHOTOSOURCETYPE
FROM workspace.prxbi_ts_sep.WC_MERCURY_BADGE_TS AMERCURY_BADGE_TS
INNER JOIN (
    SELECT
        AMERCURY_BADGE_TS_1.ID AS ID,
        MAX(AMERCURY_BADGE_TS_1.INT_INSERT_DATE) AS INT_INSERT_DATE,
        MAX(AMERCURY_BADGE_TS_1.VERSIONNUMBER) AS VERSIONNUMBER
    FROM workspace.prxbi_ts_sep.WC_MERCURY_BADGE_TS AMERCURY_BADGE_TS_1
    WHERE AMERCURY_BADGE_TS_1.INT_INSERT_DATE > (SELECT etl_last_extract_time FROM V_ETL_LAST_EXTRACT_TIME)
      AND AMERCURY_BADGE_TS_1.INT_INSERT_DATE <= (SELECT etl_current_extract_time FROM V_ETL_CURRENT_EXTRACT_TIME)
    GROUP BY AMERCURY_BADGE_TS_1.ID
) AMERCURY_BADGE_TS_2
    ON AMERCURY_BADGE_TS.INT_INSERT_DATE = AMERCURY_BADGE_TS_2.INT_INSERT_DATE
   AND AMERCURY_BADGE_TS.VERSIONNUMBER = AMERCURY_BADGE_TS_2.VERSIONNUMBER
   AND AMERCURY_BADGE_TS.ID = AMERCURY_BADGE_TS_2.ID
WHERE (1=1)

---
## Step 3: I$ Flow Table - Column Mapping, Product Lookup, and Change Detection
Map C$ columns to target names, LEFT JOIN to WC_BADGE_PRODUCT_D for PACKAGE_NAME lookup,
and apply NOT EXISTS change detection combined with IND_UPDATE logic:
- `'I'` = New row (not in target)
- `'U'` = Existing row with changed columns
- Rows matching all columns in target are excluded via WHERE filter

In [ ]:
%sql
-- Step 3: I$ Flow Table with column mapping, product lookup, and NOT EXISTS change detection
CREATE OR REPLACE TEMP VIEW I_WC_BADGE_DETAILS_D AS
SELECT
    S.BADGE_ID,
    S.BADGE_LOCATION,
    S.BADGE_TOKEN,
    S.BADGE_VERSION,
    S.CONTACT_EMAIL,
    S.CONTACT_FIRST_NAME,
    S.CONTACT_LAST_NAME,
    S.CONTACT_JOB_TITLE,
    S.CONTACT_PERSON_ID,
    S.CREATION_REG_TYPE,
    S.CREATION_TYPE,
    S.CULTURE,
    S.CUSTOMER_TYPE,
    S.EVENT_EDITION_CODE,
    S.BADGE_UPDATE_FLG,
    S.MARKETING_PREF_PROMPT,
    S.ORG_NAME,
    S.ORG_CITY,
    S.ORG_COUNTRY,
    S.ORG_ID,
    S.ORG_STATE,
    S.PARTICIPATING_ORG_ID,
    S.PRODUCT_CODE,
    S.QR_CODE,
    S.REGISTRATION_ID,
    S.STATUS,
    S.STAFF_COMPANY_NAME,
    S.STAFF_COMPANY_ADDR,
    S.STAFF_PHONE_NUM,
    S.STAFF_REPORTING,
    S.STAFF_STANDS,
    S.STAFF_USER_ACCESS,
    S.VERSION_NUM,
    S.INTEGRATION_ID,
    S.DATASOURCE_NUM_ID,
    S.MOBILEPHONE,
    S.FIRSTSCANNEDDATE,
    S.LASTPRINTEDDATE,
    S.FIRSTSCANNEDDATE_FLG,
    S.LASTPRINTEDDATE_FLG,
    S.ACCESSVALIDITYMODIFIEDDATE,
    S.CREATEDDATE,
    S.COMPANYPRODUCTCODE,
    S.PAYMENTSTATUS,
    S.PHOTOKEY,
    S.PHOTOSOURCE,
    S.PHOTOSOURCETYPE,
    S.PACKAGE_NAME,
    CASE
        WHEN T.ROW_WID IS NOT NULL THEN 'U'
        ELSE 'I'
    END AS IND_UPDATE
FROM (
    -- Column mapping from C$ + product lookup
    SELECT
        JOIN1_A.ID AS BADGE_ID,
        JOIN1_A.BADGELOCATION AS BADGE_LOCATION,
        JOIN1_A.BADGETOKEN AS BADGE_TOKEN,
        JOIN1_A.BADGEVERSION AS BADGE_VERSION,
        JOIN1_A.CONTACTEMAIL AS CONTACT_EMAIL,
        JOIN1_A.CONTACTFIRSTNAME AS CONTACT_FIRST_NAME,
        JOIN1_A.CONTACTLASTNAME AS CONTACT_LAST_NAME,
        JOIN1_A.CONTACTJOBTITLE AS CONTACT_JOB_TITLE,
        JOIN1_A.CONTACTPERSONRXMASTERID AS CONTACT_PERSON_ID,
        JOIN1_A.CREATEDBYREGISTRATIONTYPE AS CREATION_REG_TYPE,
        JOIN1_A.CREATEDBYTYPE AS CREATION_TYPE,
        JOIN1_A.CULTURE AS CULTURE,
        JOIN1_A.CUSTOMERTYPE AS CUSTOMER_TYPE,
        JOIN1_A.EVENTEDITIONGBSCODE AS EVENT_EDITION_CODE,
        JOIN1_A.ISBADGEUPDATE AS BADGE_UPDATE_FLG,
        JOIN1_A.MARKETINGPREFERENCESPROMPTREQU AS MARKETING_PREF_PROMPT,
        JOIN1_A.ORGANISATIONDISPLAYNAME AS ORG_NAME,
        JOIN1_A.ORGANISATIONCITY AS ORG_CITY,
        JOIN1_A.ORGANISATIONCOUNTRYCODE AS ORG_COUNTRY,
        JOIN1_A.ORGANISATIONRXMASTERID AS ORG_ID,
        JOIN1_A.ORGANISATIONSTATE AS ORG_STATE,
        JOIN1_A.PARTICIPATINGORGANISATIONID AS PARTICIPATING_ORG_ID,
        JOIN1_A.PRODUCTCODE AS PRODUCT_CODE,
        JOIN1_A.QRCODECONTENT AS QR_CODE,
        JOIN1_A.REGISTRATIONID AS REGISTRATION_ID,
        JOIN1_A.STATUS AS STATUS,
        JOIN1_A.SUPPORTSTAFFCOMPANYNAME AS STAFF_COMPANY_NAME,
        JOIN1_A.SUPPORTSTAFFCOMPANYADDRESS AS STAFF_COMPANY_ADDR,
        JOIN1_A.SUPPORTSTAFFMOBILEPHONE AS STAFF_PHONE_NUM,
        JOIN1_A.SUPPORTSTAFFREPORTSTO AS STAFF_REPORTING,
        JOIN1_A.SUPPORTSTAFFSTANDS AS STAFF_STANDS,
        JOIN1_A.SUPPORTSTAFFUSERACCESS AS STAFF_USER_ACCESS,
        JOIN1_A.VERSIONNUMBER AS VERSION_NUM,
        JOIN1_A.ID AS INTEGRATION_ID,
        380 AS DATASOURCE_NUM_ID,
        JOIN1_A.MOBILEPHONE AS MOBILEPHONE,
        JOIN1_A.FIRSTSCANNEDDATE AS FIRSTSCANNEDDATE,
        JOIN1_A.LASTPRINTEDDATE AS LASTPRINTEDDATE,
        CASE WHEN JOIN1_A.FIRSTSCANNEDDATE IS NOT NULL THEN 'Y' ELSE 'N' END AS FIRSTSCANNEDDATE_FLG,
        CASE WHEN JOIN1_A.LASTPRINTEDDATE IS NOT NULL THEN 'Y' ELSE 'N' END AS LASTPRINTEDDATE_FLG,
        JOIN1_A.ACCESSVALIDITYMODIFIEDDATE AS ACCESSVALIDITYMODIFIEDDATE,
        JOIN1_A.CREATEDDATE AS CREATEDDATE,
        JOIN1_A.COMPANYPRODUCTCODE AS COMPANYPRODUCTCODE,
        JOIN1_A.PAYMENTSTATUS AS PAYMENTSTATUS,
        JOIN1_A.PHOTOKEY AS PHOTOKEY,
        JOIN1_A.PHOTOSOURCE AS PHOTOSOURCE,
        JOIN1_A.PHOTOSOURCETYPE AS PHOTOSOURCETYPE,
        WC_BADGE_PRODUCT_D_2.NAME_1 AS PACKAGE_NAME
    FROM C_WC_BADGE_DETAILS JOIN1_A
    LEFT OUTER JOIN (
        SELECT
            WC_BADGE_PRODUCT_D_1.ID,
            WC_BADGE_PRODUCT_D_1.SKU,
            WC_BADGE_PRODUCT_D_1.NAME,
            WC_BADGE_PRODUCT_D_1.COL,
            WC_BADGE_PRODUCT_D_1.SKU_1,
            WC_BADGE_PRODUCT_D_1.NAME_1
        FROM (
            SELECT
                WC_BADGE_PRODUCT_D.ID,
                WC_BADGE_PRODUCT_D.SKU,
                WC_BADGE_PRODUCT_D.NAME,
                RANK() OVER (PARTITION BY WC_BADGE_PRODUCT_D.SKU ORDER BY WC_BADGE_PRODUCT_D.ID DESC) AS COL,
                WC_BADGE_PRODUCT_D.SKU AS SKU_1,
                WC_BADGE_PRODUCT_D.NAME AS NAME_1
            FROM workspace.prxbi_dw_sep.WC_BADGE_PRODUCT_D WC_BADGE_PRODUCT_D
        ) WC_BADGE_PRODUCT_D_1
        WHERE WC_BADGE_PRODUCT_D_1.COL = 1
    ) WC_BADGE_PRODUCT_D_2
        ON JOIN1_A.PRODUCTCODE = WC_BADGE_PRODUCT_D_2.SKU_1
) S
-- NOT EXISTS change detection: exclude rows that match ALL columns in target
LEFT OUTER JOIN workspace.prxbi_dw_sep.WC_BADGE_DETAILS_D T
    ON T.INTEGRATION_ID = S.INTEGRATION_ID
   AND T.DATASOURCE_NUM_ID = S.DATASOURCE_NUM_ID
WHERE NOT (
    T.ROW_WID IS NOT NULL
    AND ((T.BADGE_ID = S.BADGE_ID) OR (T.BADGE_ID IS NULL AND S.BADGE_ID IS NULL))
    AND ((T.BADGE_LOCATION = S.BADGE_LOCATION) OR (T.BADGE_LOCATION IS NULL AND S.BADGE_LOCATION IS NULL))
    AND ((T.BADGE_TOKEN = S.BADGE_TOKEN) OR (T.BADGE_TOKEN IS NULL AND S.BADGE_TOKEN IS NULL))
    AND ((T.BADGE_VERSION = S.BADGE_VERSION) OR (T.BADGE_VERSION IS NULL AND S.BADGE_VERSION IS NULL))
    AND ((T.CONTACT_EMAIL = S.CONTACT_EMAIL) OR (T.CONTACT_EMAIL IS NULL AND S.CONTACT_EMAIL IS NULL))
    AND ((T.CONTACT_FIRST_NAME = S.CONTACT_FIRST_NAME) OR (T.CONTACT_FIRST_NAME IS NULL AND S.CONTACT_FIRST_NAME IS NULL))
    AND ((T.CONTACT_LAST_NAME = S.CONTACT_LAST_NAME) OR (T.CONTACT_LAST_NAME IS NULL AND S.CONTACT_LAST_NAME IS NULL))
    AND ((T.CONTACT_JOB_TITLE = S.CONTACT_JOB_TITLE) OR (T.CONTACT_JOB_TITLE IS NULL AND S.CONTACT_JOB_TITLE IS NULL))
    AND ((T.CONTACT_PERSON_ID = S.CONTACT_PERSON_ID) OR (T.CONTACT_PERSON_ID IS NULL AND S.CONTACT_PERSON_ID IS NULL))
    AND ((T.CREATION_REG_TYPE = S.CREATION_REG_TYPE) OR (T.CREATION_REG_TYPE IS NULL AND S.CREATION_REG_TYPE IS NULL))
    AND ((T.CREATION_TYPE = S.CREATION_TYPE) OR (T.CREATION_TYPE IS NULL AND S.CREATION_TYPE IS NULL))
    AND ((T.CULTURE = S.CULTURE) OR (T.CULTURE IS NULL AND S.CULTURE IS NULL))
    AND ((T.CUSTOMER_TYPE = S.CUSTOMER_TYPE) OR (T.CUSTOMER_TYPE IS NULL AND S.CUSTOMER_TYPE IS NULL))
    AND ((T.EVENT_EDITION_CODE = S.EVENT_EDITION_CODE) OR (T.EVENT_EDITION_CODE IS NULL AND S.EVENT_EDITION_CODE IS NULL))
    AND ((T.BADGE_UPDATE_FLG = S.BADGE_UPDATE_FLG) OR (T.BADGE_UPDATE_FLG IS NULL AND S.BADGE_UPDATE_FLG IS NULL))
    AND ((T.MARKETING_PREF_PROMPT = S.MARKETING_PREF_PROMPT) OR (T.MARKETING_PREF_PROMPT IS NULL AND S.MARKETING_PREF_PROMPT IS NULL))
    AND ((T.ORG_NAME = S.ORG_NAME) OR (T.ORG_NAME IS NULL AND S.ORG_NAME IS NULL))
    AND ((T.ORG_CITY = S.ORG_CITY) OR (T.ORG_CITY IS NULL AND S.ORG_CITY IS NULL))
    AND ((T.ORG_COUNTRY = S.ORG_COUNTRY) OR (T.ORG_COUNTRY IS NULL AND S.ORG_COUNTRY IS NULL))
    AND ((T.ORG_ID = S.ORG_ID) OR (T.ORG_ID IS NULL AND S.ORG_ID IS NULL))
    AND ((T.ORG_STATE = S.ORG_STATE) OR (T.ORG_STATE IS NULL AND S.ORG_STATE IS NULL))
    AND ((T.PARTICIPATING_ORG_ID = S.PARTICIPATING_ORG_ID) OR (T.PARTICIPATING_ORG_ID IS NULL AND S.PARTICIPATING_ORG_ID IS NULL))
    AND ((T.PRODUCT_CODE = S.PRODUCT_CODE) OR (T.PRODUCT_CODE IS NULL AND S.PRODUCT_CODE IS NULL))
    AND ((T.QR_CODE = S.QR_CODE) OR (T.QR_CODE IS NULL AND S.QR_CODE IS NULL))
    AND ((T.REGISTRATION_ID = S.REGISTRATION_ID) OR (T.REGISTRATION_ID IS NULL AND S.REGISTRATION_ID IS NULL))
    AND ((T.STATUS = S.STATUS) OR (T.STATUS IS NULL AND S.STATUS IS NULL))
    AND ((T.STAFF_COMPANY_NAME = S.STAFF_COMPANY_NAME) OR (T.STAFF_COMPANY_NAME IS NULL AND S.STAFF_COMPANY_NAME IS NULL))
    AND ((T.STAFF_COMPANY_ADDR = S.STAFF_COMPANY_ADDR) OR (T.STAFF_COMPANY_ADDR IS NULL AND S.STAFF_COMPANY_ADDR IS NULL))
    AND ((T.STAFF_PHONE_NUM = S.STAFF_PHONE_NUM) OR (T.STAFF_PHONE_NUM IS NULL AND S.STAFF_PHONE_NUM IS NULL))
    AND ((T.STAFF_REPORTING = S.STAFF_REPORTING) OR (T.STAFF_REPORTING IS NULL AND S.STAFF_REPORTING IS NULL))
    AND ((T.STAFF_STANDS = S.STAFF_STANDS) OR (T.STAFF_STANDS IS NULL AND S.STAFF_STANDS IS NULL))
    AND ((T.STAFF_USER_ACCESS = S.STAFF_USER_ACCESS) OR (T.STAFF_USER_ACCESS IS NULL AND S.STAFF_USER_ACCESS IS NULL))
    AND ((T.VERSION_NUM = S.VERSION_NUM) OR (T.VERSION_NUM IS NULL AND S.VERSION_NUM IS NULL))
    AND ((T.MOBILEPHONE = S.MOBILEPHONE) OR (T.MOBILEPHONE IS NULL AND S.MOBILEPHONE IS NULL))
    AND ((T.FIRSTSCANNEDDATE = S.FIRSTSCANNEDDATE) OR (T.FIRSTSCANNEDDATE IS NULL AND S.FIRSTSCANNEDDATE IS NULL))
    AND ((T.LASTPRINTEDDATE = S.LASTPRINTEDDATE) OR (T.LASTPRINTEDDATE IS NULL AND S.LASTPRINTEDDATE IS NULL))
    AND ((T.FIRSTSCANNEDDATE_FLG = S.FIRSTSCANNEDDATE_FLG) OR (T.FIRSTSCANNEDDATE_FLG IS NULL AND S.FIRSTSCANNEDDATE_FLG IS NULL))
    AND ((T.LASTPRINTEDDATE_FLG = S.LASTPRINTEDDATE_FLG) OR (T.LASTPRINTEDDATE_FLG IS NULL AND S.LASTPRINTEDDATE_FLG IS NULL))
    AND ((T.ACCESSVALIDITYMODIFIEDDATE = S.ACCESSVALIDITYMODIFIEDDATE) OR (T.ACCESSVALIDITYMODIFIEDDATE IS NULL AND S.ACCESSVALIDITYMODIFIEDDATE IS NULL))
    AND ((T.CREATEDDATE = S.CREATEDDATE) OR (T.CREATEDDATE IS NULL AND S.CREATEDDATE IS NULL))
    AND ((T.COMPANYPRODUCTCODE = S.COMPANYPRODUCTCODE) OR (T.COMPANYPRODUCTCODE IS NULL AND S.COMPANYPRODUCTCODE IS NULL))
    AND ((T.PAYMENTSTATUS = S.PAYMENTSTATUS) OR (T.PAYMENTSTATUS IS NULL AND S.PAYMENTSTATUS IS NULL))
    AND ((T.PHOTOKEY = S.PHOTOKEY) OR (T.PHOTOKEY IS NULL AND S.PHOTOKEY IS NULL))
    AND ((T.PHOTOSOURCE = S.PHOTOSOURCE) OR (T.PHOTOSOURCE IS NULL AND S.PHOTOSOURCE IS NULL))
    AND ((T.PHOTOSOURCETYPE = S.PHOTOSOURCETYPE) OR (T.PHOTOSOURCETYPE IS NULL AND S.PHOTOSOURCETYPE IS NULL))
    AND ((T.PACKAGE_NAME = S.PACKAGE_NAME) OR (T.PACKAGE_NAME IS NULL AND S.PACKAGE_NAME IS NULL))
)

---
## Step 4: MERGE for Updates
Update existing rows in `WC_BADGE_DETAILS_D` where `IND_UPDATE = 'U'`.

In [ ]:
%sql
-- Step 4: MERGE for Updates (IND_UPDATE = 'U')
MERGE INTO workspace.prxbi_dw_sep.WC_BADGE_DETAILS_D AS T
USING (
    SELECT * FROM I_WC_BADGE_DETAILS_D WHERE IND_UPDATE = 'U'
) S
ON T.INTEGRATION_ID = S.INTEGRATION_ID
   AND T.DATASOURCE_NUM_ID = S.DATASOURCE_NUM_ID
WHEN MATCHED THEN UPDATE SET
    T.BADGE_ID = S.BADGE_ID,
    T.BADGE_LOCATION = S.BADGE_LOCATION,
    T.BADGE_TOKEN = S.BADGE_TOKEN,
    T.BADGE_VERSION = S.BADGE_VERSION,
    T.CONTACT_EMAIL = S.CONTACT_EMAIL,
    T.CONTACT_FIRST_NAME = S.CONTACT_FIRST_NAME,
    T.CONTACT_LAST_NAME = S.CONTACT_LAST_NAME,
    T.CONTACT_JOB_TITLE = S.CONTACT_JOB_TITLE,
    T.CONTACT_PERSON_ID = S.CONTACT_PERSON_ID,
    T.CREATION_REG_TYPE = S.CREATION_REG_TYPE,
    T.CREATION_TYPE = S.CREATION_TYPE,
    T.CULTURE = S.CULTURE,
    T.CUSTOMER_TYPE = S.CUSTOMER_TYPE,
    T.EVENT_EDITION_CODE = S.EVENT_EDITION_CODE,
    T.BADGE_UPDATE_FLG = S.BADGE_UPDATE_FLG,
    T.MARKETING_PREF_PROMPT = S.MARKETING_PREF_PROMPT,
    T.ORG_NAME = S.ORG_NAME,
    T.ORG_CITY = S.ORG_CITY,
    T.ORG_COUNTRY = S.ORG_COUNTRY,
    T.ORG_ID = S.ORG_ID,
    T.ORG_STATE = S.ORG_STATE,
    T.PARTICIPATING_ORG_ID = S.PARTICIPATING_ORG_ID,
    T.PRODUCT_CODE = S.PRODUCT_CODE,
    T.QR_CODE = S.QR_CODE,
    T.REGISTRATION_ID = S.REGISTRATION_ID,
    T.STATUS = S.STATUS,
    T.STAFF_COMPANY_NAME = S.STAFF_COMPANY_NAME,
    T.STAFF_COMPANY_ADDR = S.STAFF_COMPANY_ADDR,
    T.STAFF_PHONE_NUM = S.STAFF_PHONE_NUM,
    T.STAFF_REPORTING = S.STAFF_REPORTING,
    T.STAFF_STANDS = S.STAFF_STANDS,
    T.STAFF_USER_ACCESS = S.STAFF_USER_ACCESS,
    T.VERSION_NUM = S.VERSION_NUM,
    T.MOBILEPHONE = S.MOBILEPHONE,
    T.FIRSTSCANNEDDATE = S.FIRSTSCANNEDDATE,
    T.LASTPRINTEDDATE = S.LASTPRINTEDDATE,
    T.FIRSTSCANNEDDATE_FLG = S.FIRSTSCANNEDDATE_FLG,
    T.LASTPRINTEDDATE_FLG = S.LASTPRINTEDDATE_FLG,
    T.ACCESSVALIDITYMODIFIEDDATE = S.ACCESSVALIDITYMODIFIEDDATE,
    T.CREATEDDATE = S.CREATEDDATE,
    T.COMPANYPRODUCTCODE = S.COMPANYPRODUCTCODE,
    T.PAYMENTSTATUS = S.PAYMENTSTATUS,
    T.PHOTOKEY = S.PHOTOKEY,
    T.PHOTOSOURCE = S.PHOTOSOURCE,
    T.PHOTOSOURCETYPE = S.PHOTOSOURCETYPE,
    T.PACKAGE_NAME = S.PACKAGE_NAME,
    T.W_UPDATE_DT = current_timestamp(),
    T.ETL_PROC_WID = (SELECT ROW_WID FROM V_ETL_PROC_WID)

---
## Step 5: INSERT for New Rows
Insert new rows into `WC_BADGE_DETAILS_D` where `IND_UPDATE = 'I'`, generating `ROW_WID` using `row_number() + MAX(ROW_WID)`.

In [ ]:
%sql
-- Step 5: INSERT new rows (IND_UPDATE = 'I') with ROW_WID generation
INSERT INTO workspace.prxbi_dw_sep.WC_BADGE_DETAILS_D
(
    ROW_WID,
    BADGE_ID,
    BADGE_LOCATION,
    BADGE_TOKEN,
    BADGE_VERSION,
    CONTACT_EMAIL,
    CONTACT_FIRST_NAME,
    CONTACT_LAST_NAME,
    CONTACT_JOB_TITLE,
    CONTACT_PERSON_ID,
    CREATION_REG_TYPE,
    CREATION_TYPE,
    CULTURE,
    CUSTOMER_TYPE,
    EVENT_EDITION_CODE,
    BADGE_UPDATE_FLG,
    MARKETING_PREF_PROMPT,
    ORG_NAME,
    ORG_CITY,
    ORG_COUNTRY,
    ORG_ID,
    ORG_STATE,
    PARTICIPATING_ORG_ID,
    PRODUCT_CODE,
    QR_CODE,
    REGISTRATION_ID,
    STATUS,
    STAFF_COMPANY_NAME,
    STAFF_COMPANY_ADDR,
    STAFF_PHONE_NUM,
    STAFF_REPORTING,
    STAFF_STANDS,
    STAFF_USER_ACCESS,
    VERSION_NUM,
    INTEGRATION_ID,
    DATASOURCE_NUM_ID,
    MOBILEPHONE,
    FIRSTSCANNEDDATE,
    LASTPRINTEDDATE,
    FIRSTSCANNEDDATE_FLG,
    LASTPRINTEDDATE_FLG,
    ACCESSVALIDITYMODIFIEDDATE,
    CREATEDDATE,
    COMPANYPRODUCTCODE,
    PAYMENTSTATUS,
    PHOTOKEY,
    PHOTOSOURCE,
    PHOTOSOURCETYPE,
    PACKAGE_NAME,
    W_INSERT_DT,
    W_UPDATE_DT,
    ETL_PROC_WID
)
SELECT
    row_number() OVER (ORDER BY INTEGRATION_ID) + COALESCE((SELECT MAX(ROW_WID) FROM workspace.prxbi_dw_sep.WC_BADGE_DETAILS_D), 0) AS ROW_WID,
    BADGE_ID,
    BADGE_LOCATION,
    BADGE_TOKEN,
    BADGE_VERSION,
    CONTACT_EMAIL,
    CONTACT_FIRST_NAME,
    CONTACT_LAST_NAME,
    CONTACT_JOB_TITLE,
    CONTACT_PERSON_ID,
    CREATION_REG_TYPE,
    CREATION_TYPE,
    CULTURE,
    CUSTOMER_TYPE,
    EVENT_EDITION_CODE,
    BADGE_UPDATE_FLG,
    MARKETING_PREF_PROMPT,
    ORG_NAME,
    ORG_CITY,
    ORG_COUNTRY,
    ORG_ID,
    ORG_STATE,
    PARTICIPATING_ORG_ID,
    PRODUCT_CODE,
    QR_CODE,
    REGISTRATION_ID,
    STATUS,
    STAFF_COMPANY_NAME,
    STAFF_COMPANY_ADDR,
    STAFF_PHONE_NUM,
    STAFF_REPORTING,
    STAFF_STANDS,
    STAFF_USER_ACCESS,
    VERSION_NUM,
    INTEGRATION_ID,
    DATASOURCE_NUM_ID,
    MOBILEPHONE,
    FIRSTSCANNEDDATE,
    LASTPRINTEDDATE,
    FIRSTSCANNEDDATE_FLG,
    LASTPRINTEDDATE_FLG,
    ACCESSVALIDITYMODIFIEDDATE,
    CREATEDDATE,
    COMPANYPRODUCTCODE,
    PAYMENTSTATUS,
    PHOTOKEY,
    PHOTOSOURCE,
    PHOTOSOURCETYPE,
    PACKAGE_NAME,
    current_timestamp() AS W_INSERT_DT,
    current_timestamp() AS W_UPDATE_DT,
    (SELECT ROW_WID FROM V_ETL_PROC_WID) AS ETL_PROC_WID
FROM I_WC_BADGE_DETAILS_D
WHERE IND_UPDATE = 'I'

---
## Step 6: Update ETL Tracking
Update `WC_ETL_PARAMETERS` to record the current extract time as the new last extract time.

In [ ]:
%sql
-- Step 6: Update ETL tracking parameters
UPDATE workspace.prxbi_dw_sep.wc_etl_parameters
SET etl_last_extract_time = current_timestamp()
WHERE ETL_JOB_TYPE = 'EOD'

---
## Step 7: Cleanup - Drop Temp Views

In [ ]:
%sql
-- Step 7: Cleanup temp views
DROP VIEW IF EXISTS V_ETL_LAST_EXTRACT_TIME;
DROP VIEW IF EXISTS V_ETL_CURRENT_EXTRACT_TIME;
DROP VIEW IF EXISTS V_ETL_PROC_WID;
DROP VIEW IF EXISTS C_WC_BADGE_DETAILS;
DROP VIEW IF EXISTS I_WC_BADGE_DETAILS_D